# Neural Search 101: Build a Semantic Search Service with Qdrant + FastEmbed

This notebook follows the [Neural Search 101 tutorial](https://qdrant.tech/articles/neural-search-tutorial/) step by step.

We will:
1. Encode startup descriptions into vectors using **FastEmbed** (no separate encoding step needed - Qdrant's client embeds text for us).
2. Store and index those vectors in **Qdrant**.
3. Build a simple `NeuralSearcher` class to query them by meaning, not just keywords.
4. Add a payload filter to narrow results (e.g. by city).

This notebook connects to a **Qdrant Cloud** cluster, so the collection you create here persists and can be queried again later - including from the FastAPI service described in the article (see the note in Step 3).

## Step 1: Install dependencies

`qdrant-client[fastembed]` bundles the FastEmbed library, so text gets embedded automatically whenever we upload or query - no manual encoding step required.

In [1]:
!pip install -q "qdrant-client[fastembed]>=1.14.2" pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 15.6 MB/s eta 0:00:00


## Step 2: Download the sample dataset

We'll search over startup descriptions from [startups-list.com](https://www.startups-list.com/). Each record has a name, description, location, and image.

In [2]:
!wget -q https://storage.googleapis.com/generall-shared-data/startups_demo.json

import pandas as pd

# Quick peek at the data - one JSON object per line
df = pd.read_json("startups_demo.json", lines=True)
print(f"Loaded {len(df)} startup records")
df.head()

Loaded 40474 startup records


,name,images,alt,description,link,city
0,SaferCodes,https://safer.codes/img/brand/logo-icon.png,SaferCodes Logo QR codes generator system form...,QR codes systems for COVID-19.\nSimple tools f...,https://safer.codes,Chicago
1,Human Practice,https://d1qb2nb5cznatu.cloudfront.net/startups...,Human Practice - health care information tech...,Point-of-care word of mouth\nPreferral is a mo...,http://humanpractice.com,Chicago
2,StyleSeek,https://d1qb2nb5cznatu.cloudfront.net/startups...,StyleSeek - e-commerce fashion mass customiza...,Personalized e-commerce for lifestyle products...,http://styleseek.com,Chicago
3,Scout,https://d1qb2nb5cznatu.cloudfront.net/startups...,Scout - security consumer electronics interne...,Hassle-free Home Security\nScout is a self-ins...,http://www.scoutalarm.com,Chicago
4,Invitation codes,https://invitation.codes/img/inv-brand-fb3.png,Invitation App - Share referral codes community,The referral community\nInvitation App is a so...,https://invitation.codes,Chicago


## Step 3: Connect to Qdrant

This notebook connects to a **Qdrant Cloud** cluster. If you don't have one yet, create a free cluster at [cloud.qdrant.io](https://cloud.qdrant.io/) and grab its **URL** and an **API key** from the cluster dashboard.

The safest way to provide them in Colab is via **Secrets** (the key icon in the left sidebar): add `QDRANT_URL` and `QDRANT_API_KEY`, then toggle "Notebook access" on for each. The cell below reads them from there.

> Running locally instead of Colab? Just set the two variables directly, or via environment variables, instead of using `userdata`.

In [3]:
from qdrant_client import QdrantClient, models
from google.colab import userdata

QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

## Step 4: Create a collection

We'll use `sentence-transformers/all-MiniLM-L6-v2` to encode descriptions - a fast, all-round model. Instead of hard-coding its output dimensionality, we ask the client for it directly.

In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
collection_name = "startups"

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=client.get_embedding_size(model_name),
            distance=models.Distance.COSINE,
        ),
    )

print(f"Vector size for {model_name}: {client.get_embedding_size(model_name)}")

Vector size for sentence-transformers/all-MiniLM-L6-v2: 384


## Step 5: Upload data to Qdrant

We wrap each description in `models.Document`. This tells Qdrant *what* to embed and *which* model to use - the actual embedding happens automatically as part of `upload_collection`, batch by batch.

> **Note:** encoding ~40k descriptions on a CPU can take a while. To keep this notebook quick to run, we only use the first `NUM_ROWS` records below - increase or remove the slice to run over the full dataset.

In [5]:
NUM_ROWS = 2000  # set to None to use the full dataset

records = df.to_dict("records") if NUM_ROWS is None else df.head(NUM_ROWS).to_dict("records")

payload = []
vectors = []

for record in records:
    payload.append(record)
    vectors.append(models.Document(text=record["description"], model=model_name))

print(f"Prepared {len(vectors)} documents for upload")

Prepared 2000 documents for upload


In [6]:
client.upload_collection(
    collection_name=collection_name,
    vectors=vectors,
    payload=payload,
    ids=None,  # Vector ids are assigned automatically
    batch_size=256,  # How many vectors are embedded and uploaded per request
)

print(client.get_collection(collection_name))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=2000 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None), wal_config=WalCon

## Step 6: Build the search API

This mirrors the `NeuralSearcher` class from the article - it just takes an existing `QdrantClient` instead of creating a new one, so the notebook and any other service (like the FastAPI app from the article) can share the same Qdrant Cloud cluster.

In [7]:
class NeuralSearcher:
    def __init__(self, collection_name, client, model_name):
        self.collection_name = collection_name
        self.model_name = model_name
        self.qdrant_client = client

    def search(self, text: str, query_filter: models.Filter = None, limit: int = 5):
        # `models.Document` tells Qdrant to embed the query with FastEmbed,
        # using the same model that was used to embed the uploaded data
        search_result = self.qdrant_client.query_points(
            collection_name=self.collection_name,
            query=models.Document(text=text, model=self.model_name),
            query_filter=query_filter,
            limit=limit,
        ).points
        # We are interested in the payload only, not the raw vectors/scores
        return [hit.payload for hit in search_result]


neural_searcher = NeuralSearcher(
    collection_name=collection_name, client=client, model_name=model_name
)

In [8]:
# Try a query - notice none of these words need to literally appear in the results
results = neural_searcher.search("organic food delivery")
for hit in results:
    print(hit["name"], "-", hit["city"])
    print(" ", hit["description"][:150], "...")
    print()

Home Chef - Chicago
  Fresh ingredient delivery
Our mission is to make cooking fresh food at home as easy as possible.
Our chefs shop for you and plan your meals, allowing  ...

Nip - Chicago
  Food delivery simplified for the hungry and the busy ones
Nip offers a curated selection of moderately-priced meals, from the best restaurants in your ...

Zesty - San Francisco
  Healthy, delicious food. Delivered. (YC W14)
Zesty empowers companies to eat well, work happy and be awesome. Healthy, delicious food experiences from ...

Feed Earth Now - Chicago
  What probiotics do for performance in the human body, Terreplenish™ does for the soil.
Our business utilizes an innovative technology that rapidly con ...

Momentum Machines - San Francisco
  The Next Generation of Fast Food.
Momentum is revolutionizing the way food is prepared with technology that allows for gourmet quality food to be sold ...



## Step 7: Add a filter

Qdrant can combine vector search with structured payload filters. For example, restrict results to startups in a specific city.

In [10]:
client.create_payload_index(
    collection_name=collection_name,
    field_name="city",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

city_of_interest = "Berlin"

city_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="city",  # We store city information in a field of the same name
            match=models.MatchValue(value=city_of_interest),
        )
    ]
)

results = neural_searcher.search("organic food delivery", query_filter=city_filter)
for hit in results:
    print(hit["name"], "-", hit["city"])
    print(" ", hit["description"][:150], "...")
    print()

## Next steps

That's the whole neural search flow: embed, index, query, filter - all through Qdrant, with FastEmbed handling encoding automatically.

Since this notebook already uploaded to a real Qdrant Cloud cluster, the collection is ready to be queried from anywhere - including the small [FastAPI](https://fastapi.tiangolo.com/) app from the article's **Step 5: Deploy as a service** section, which wraps this same `NeuralSearcher` class behind a `/api/search` endpoint. See the full [tutorial](https://qdrant.tech/articles/neural-search-tutorial/) for that code.